In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv(r"c:/Users/MARIA JOSE/Downloads/sales.csv")
print(df)

    # Gotta simulate lead time, initial stock. Calculate safety stock and rop

     product_id       sdate region  units_sold product_category  unit_cost  \
0          1052  2023-02-03  North          18        Furniture     152.75   
1          1093  2023-04-21   West          17        Furniture    3816.39   
2          1015  2023-09-21  South          30             Food     261.56   
3          1072  2023-08-24  South          39         Clothing    4330.03   
4          1061  2023-03-24   East          13      Electronics     637.37   
..          ...         ...    ...         ...              ...        ...   
995        1010  2023-04-15  North           4             Food    4943.03   
996        1067  2023-09-07  North          37         Clothing    1754.32   
997        1018  2023-04-27  South          17         Clothing     355.72   
998        1100  2023-12-20   West          39      Electronics    3685.03   
999        1086  2023-08-16   East          48             Food    2632.58   

     unit_price  
0        267.22  
1       4209.44  
2        

In [ ]:
# AVAILABILITY PERIOD
df['sdate'] = pd.to_datetime(df['sdate'])


time_on_sale = (
    df.groupby('product_id')['sdate']
      .agg(['min', 'max'])
)

time_on_sale['time_on_sale'] = (
    (time_on_sale['max'] - time_on_sale['min']).dt.days + 1
)

units = (
    df.groupby('product_id')['units_sold']
      .sum()
)

print(units)

product_id
1001    374
1002    308
1003    255
1004    230
1005    221
       ... 
1096    341
1097    215
1098    126
1099    502
1100    239
Name: units_sold, Length: 100, dtype: int64


In [31]:
df['revenue'] = df['units_sold'] * df['unit_price']

sku_params = df.groupby('product_id').agg(
    demand_std = ('units_sold', 'std'),
    tunits_sold = ('units_sold', 'sum'),
    trevenue = ('revenue', 'sum')
   
).reset_index()

print(sku_params)


    product_id  demand_std  tunits_sold    trevenue
0         1001   14.827340          374  1012319.51
1         1002    9.612358          308   745910.34
2         1003   12.828413          255   750608.69
3         1004   14.494513          230   865859.72
4         1005   14.625010          221   657473.62
..         ...         ...          ...         ...
95        1096   12.885094          341   865465.68
96        1097   16.729050          215   738643.22
97        1098    9.539392          126   320386.09
98        1099   12.072764          502  1635859.89
99        1100   14.543995          239   653313.62

[100 rows x 4 columns]


In [ ]:
# DEMAND ADJUSTMENT TO AVAILABILITY PERIOD 
df['sdate'] = pd.to_datetime(df['sdate'])


time_on_sale = (
    df.groupby('product_id')['sdate']
      .agg(['min', 'max'])
)

time_on_sale['time_on_sale'] = (
    (time_on_sale['max'] - time_on_sale['min']).dt.days + 1
)

units = (
    df.groupby('product_id')['units_sold']
      .sum()
)

avg_dmd = units / time_on_sale['time_on_sale']

print(avg_dmd)



product_id
1001    1.214286
1002    0.933333
1003    0.739130
1004    0.861423
1005    0.754266
          ...   
1096    1.096463
1097    0.790441
1098    0.431507
1099    1.386740
1100    0.794020
Length: 100, dtype: float64


In [33]:
sku_params['avg_daily_dmd'] = (
    sku_params['tunits_sold'] /
    sku_params['product_id'].map(time_on_sale['time_on_sale'])
)
print(sku_params['avg_daily_dmd'])

0     1.214286
1     0.933333
2     0.739130
3     0.861423
4     0.754266
        ...   
95    1.096463
96    0.790441
97    0.431507
98    1.386740
99    0.794020
Name: avg_daily_dmd, Length: 100, dtype: float64


In [ ]:
# DATA SIMULATION
np.random.seed(42)

sku_params['initial_inventory'] = np.random.randint(15, 30, len(sku_params))
print(sku_params['initial_inventory'])

sku_params['lead_time_days'] = np.random.randint(15, 25, len(sku_params))
print(sku_params['lead_time_days'])

0     21
1     18
2     27
3     29
4     25
      ..
95    27
96    29
97    16
98    24
99    26
Name: initial_inventory, Length: 100, dtype: int32
0     16
1     24
2     18
3     22
4     21
      ..
95    22
96    20
97    22
98    23
99    18
Name: lead_time_days, Length: 100, dtype: int32


In [ ]:
# SS AND ROP CALCULATION
sku_params['safety_stock'] = (
    1.65 * sku_params['demand_std'] * np.sqrt(sku_params['lead_time_days'])
)
print(round(sku_params['safety_stock']))

0      98.0
1      78.0
2      90.0
3     112.0
4     111.0
      ...  
95    100.0
96    123.0
97     74.0
98     96.0
99    102.0
Name: safety_stock, Length: 100, dtype: float64


In [36]:
sku_params['rop'] = (
   sku_params['avg_daily_dmd'] * sku_params['lead_time_days'] + sku_params['safety_stock']
)
print(round(sku_params['rop']))

0     117.0
1     100.0
2     103.0
3     131.0
4     126.0
      ...  
95    124.0
96    139.0
97     83.0
98    127.0
99    116.0
Name: rop, Length: 100, dtype: float64


In [37]:
sku_params['days_cover'] = (
    sku_params['initial_inventory'] / sku_params['avg_daily_dmd']
)
print(sku_params['days_cover'])

0     17.294118
1     19.285714
2     36.529412
3     33.665217
4     33.144796
        ...    
95    24.624633
96    36.688372
97    37.079365
98    17.306773
99    32.744770
Name: days_cover, Length: 100, dtype: float64


In [ ]:
#RISK FLAG 
sku_params['at_risk'] = (
    sku_params['days_cover'] < sku_params['lead_time_days']
)
print(sku_params['at_risk'])

0     False
1      True
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98     True
99    False
Name: at_risk, Length: 100, dtype: bool


In [39]:
sku_params['rop_close'] = (
    sku_params['initial_inventory'] - sku_params['rop']
)
print(sku_params['rop_close'])

0     -96.289013
1     -82.099733
2     -76.107821
3    -102.127044
4    -101.422944
         ...    
95    -96.842324
96   -110.252890
97    -67.320280
98   -103.428283
99    -90.105515
Name: rop_close, Length: 100, dtype: float64


In [40]:
sku_params['no_stock_days'] = (
    sku_params['lead_time_days'] - sku_params['days_cover']
).clip(lower=0)

print(sku_params['no_stock_days'])

0     0.000000
1     4.714286
2     0.000000
3     0.000000
4     0.000000
        ...   
95    0.000000
96    0.000000
97    0.000000
98    5.693227
99    0.000000
Name: no_stock_days, Length: 100, dtype: float64


In [41]:
sku_params['dmd_lost'] = (
    sku_params['no_stock_days'] * sku_params['avg_daily_dmd']
).clip(lower=0)

print(sku_params['dmd_lost'])



0     0.000000
1     4.400000
2     0.000000
3     0.000000
4     0.000000
        ...   
95    0.000000
96    0.000000
97    0.000000
98    7.895028
99    0.000000
Name: dmd_lost, Length: 100, dtype: float64


In [42]:
sku_params['avg_price'] = (
    sku_params['trevenue'] / sku_params['tunits_sold']

)

print(sku_params['avg_price'])

0     2706.736658
1     2421.786818
2     2943.563490
3     3764.607478
4     2974.993756
         ...     
95    2538.022522
96    3435.549860
97    2542.746746
98    3258.685040
99    2733.529791
Name: avg_price, Length: 100, dtype: float64


In [43]:
# FINANCIAL IMPACT

sku_params['revenue_risk'] = (
    sku_params['dmd_lost'] * sku_params['avg_price']
).clip(lower=0)

print(sku_params['revenue_risk'])
print(sum(sku_params['revenue_risk']))

0         0.000000
1     10655.862000
2         0.000000
3         0.000000
4         0.000000
          ...     
95        0.000000
96        0.000000
97        0.000000
98    25727.408408
99        0.000000
Name: revenue_risk, Length: 100, dtype: float64
378957.1954752134


In [44]:
sku_params['avg_daily_revenue'] = (
    sku_params['avg_daily_dmd'] * sku_params['avg_price']
)
print(sku_params['avg_daily_revenue'])

0     3286.751656
1     2260.334364
2     2175.677362
3     3242.920300
4     2243.937270
         ...     
95    2782.847846
96    2715.600074
97    1097.212637
98    4518.949972
99    2170.477143
Name: avg_daily_revenue, Length: 100, dtype: float64


In [45]:
trevrisk = sku_params['revenue_risk'].sum()
print(trevrisk)

sku_params['sku_risk_pctg'] = (
    sku_params['revenue_risk'] / trevrisk) * 100

print(sku_params['sku_risk_pctg'])


378957.19547521335
0     0.000000
1     2.811891
2     0.000000
3     0.000000
4     0.000000
        ...   
95    0.000000
96    0.000000
97    0.000000
98    6.789001
99    0.000000
Name: sku_risk_pctg, Length: 100, dtype: float64


In [46]:
sku_params['ac_risk_pct'] = sku_params['sku_risk_pctg'].cumsum()
sku_params['ac_risk_pct']

# 

0       0.000000
1       2.811891
2       2.811891
3       2.811891
4       2.811891
         ...    
95     93.210999
96     93.210999
97     93.210999
98    100.000000
99    100.000000
Name: ac_risk_pct, Length: 100, dtype: float64

In [47]:
# PRIORITY RANKING
sku_params['prior'] = (
    sku_params['revenue_risk'] * 0.5 +
    sku_params['tunits_sold'] * 0.3 + 
    sku_params['no_stock_days'] * 0.2
)

print(sku_params['prior'])

0       112.200000
1      5421.273857
2        76.500000
3        69.000000
4        66.300000
          ...     
95      102.300000
96       64.500000
97       37.800000
98    13015.442850
99       71.700000
Name: prior, Length: 100, dtype: float64


In [51]:
    # READJUSTMENT OF INITIAL INVENTORY BCS IT'S TOO LOW

sku_params['initial_inventory_op'] = (
    (sku_params['avg_daily_dmd'] * sku_params['lead_time_days'] * 2)
    + sku_params['safety_stock']
)

print(sku_params[['initial_inventory_op', 'initial_inventory', 'rop', 'safety_stock']]
    )



    initial_inventory_op  initial_inventory         rop  safety_stock
0             136.717584                 21  117.289013     97.860441
1             122.499733                 18  100.099733     77.699733
2             116.412169                 27  103.107821     89.803473
3             150.078354                 29  131.127044    112.175733
4             142.262534                 25  126.422944    110.583353
..                   ...                ...         ...           ...
95            147.964510                 27  123.842324     99.720137
96            155.061713                 29  139.252890    123.444066
97             92.813431                 16   83.320280     73.827129
98            159.323311                 24  127.428283     95.533256
99            130.397874                 26  116.105515    101.813156

[100 rows x 4 columns]


In [52]:
sku_params.to_csv('sku_params.csv', index = False)